# Healthcare Claims ETL Pipeline

**Layer:** ETL (Extract, Transform, Load)  
**Purpose:** Ingest raw Medicaid-style claims data, validate quality, apply transformations, and produce a curated dataset ready for analytics.

### Pipeline Flow
```
Raw CSV → Data Ingestion → Data Quality Checks → Transformation → Curated Output
```

### Scalability Note
This notebook uses **Python/pandas** for prototyping speed. The same logic is implemented in **PySpark** (`pyspark/claims_transformation.py`) for production-scale distributed processing on Databricks or EMR. Both versions produce identical output schemas.

In [1]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Pipeline config
RAW_DATA_PATH  = '../data/sample_claims.csv'
OUTPUT_DIR     = '../data/output'
RUN_TIMESTAMP  = datetime.utcnow().isoformat()

print(f'Pipeline run: {RUN_TIMESTAMP}')

Pipeline run: 2026-03-19T13:21:15.253491


C:\Users\kuan_\AppData\Local\Temp\ipykernel_8232\110651025.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TIMESTAMP  = datetime.utcnow().isoformat()


---
## Stage 1: Data Ingestion

Load raw claims data. In production this would read from GCS/S3 or a landing zone table in Databricks. All columns are loaded as strings initially — type casting happens after validation to avoid silent coercion errors.

In [2]:
# Load as strings first — type casting after validation
raw_df = pd.read_csv(RAW_DATA_PATH, dtype=str)

print(f'Rows loaded:    {len(raw_df)}')
print(f'Columns:        {len(raw_df.columns)}')
print(f'Source:         {RAW_DATA_PATH}')

raw_df.head()

Rows loaded:    20
Columns:        14
Source:         ../data/sample_claims.csv


,claim_id,member_id,state_code,plan_id,service_date,claim_type,diagnosis_code,procedure_code,billed_amount,paid_amount,claim_status,provider_id,provider_type,managed_care_flag
0,CLM0001,MBR1001,TX,MCO_TX_01,2024-01-05,IP,J18.9,99223,12500.00,9800.00,PAID,PRV5001,HOSPITAL,Y
1,CLM0002,MBR1002,TX,MCO_TX_01,2024-01-07,OP,Z00.00,99213,250.00,180.00,PAID,PRV5002,PHYSICIAN,Y
2,CLM0003,MBR1003,FL,MCO_FL_02,2024-01-08,OP,E11.9,99214,350.00,270.00,PAID,PRV5003,PHYSICIAN,Y
3,CLM0004,MBR1004,FL,MCO_FL_02,2024-01-10,IP,I21.9,99223,18000.00,14200.00,PAID,PRV5004,HOSPITAL,Y
4,CLM0005,MBR1005,CA,MCO_CA_03,2024-01-12,OP,F32.1,90837,200.00,150.00,PAID,PRV5005,BEHAVIORAL,Y


---
## Stage 2: Data Quality Checks

Quality gates that must pass before data moves to the transformation stage. In a production pipeline, rows that fail validation are written to a separate error log (not silently dropped) for audit and remediation.

**Checks performed:**
- Schema validation (required columns present)
- Null value checks on required fields
- Domain validation (claim type, status against known value sets)
- Type validation (amounts must be numeric)
- Row count validation

In [3]:
# Known valid value sets — in production these would come from a reference table
VALID_CLAIM_TYPES = {'IP', 'OP', 'LTC', 'RX'}   # Inpatient, Outpatient, Long-term Care, Pharmacy
VALID_STATUSES    = {'PAID', 'DENIED', 'PENDING', 'VOID'}
REQUIRED_COLS     = [
    'claim_id', 'member_id', 'state_code', 'service_date',
    'claim_type', 'billed_amount', 'paid_amount', 'claim_status'
]

# ── Schema validation ─────────────────────────────────────────
missing_cols = [c for c in REQUIRED_COLS if c not in raw_df.columns]
assert not missing_cols, f'Schema error — missing columns: {missing_cols}'
print(f'✓ Schema valid — all {len(REQUIRED_COLS)} required columns present')

# ── Row count baseline ────────────────────────────────────────
row_count_raw = len(raw_df)
print(f'✓ Row count (raw): {row_count_raw}')

✓ Schema valid — all 8 required columns present
✓ Row count (raw): 20


In [4]:
# ── Field-level validation ────────────────────────────────────
validation_errors = []
invalid_mask      = pd.Series(False, index=raw_df.index)

# Null checks
for col in REQUIRED_COLS:
    null_rows = raw_df[col].isna()
    for idx in raw_df[null_rows].index:
        validation_errors.append({'claim_id': raw_df.loc[idx,'claim_id'], 'error': f'Null in {col}'})
    invalid_mask |= null_rows

# Domain validation — claim_type
bad_type = ~raw_df['claim_type'].isin(VALID_CLAIM_TYPES)
for idx in raw_df[bad_type].index:
    validation_errors.append({'claim_id': raw_df.loc[idx,'claim_id'], 'error': f"Invalid claim_type: {raw_df.loc[idx,'claim_type']}"})
invalid_mask |= bad_type

# Domain validation — claim_status
bad_status = ~raw_df['claim_status'].isin(VALID_STATUSES)
for idx in raw_df[bad_status].index:
    validation_errors.append({'claim_id': raw_df.loc[idx,'claim_id'], 'error': f"Invalid claim_status: {raw_df.loc[idx,'claim_status']}"})
invalid_mask |= bad_status

# Type validation — amounts must be numeric
for col in ['billed_amount', 'paid_amount']:
    bad_num = pd.to_numeric(raw_df[col], errors='coerce').isna()
    for idx in raw_df[bad_num].index:
        validation_errors.append({'claim_id': raw_df.loc[idx,'claim_id'], 'error': f'Non-numeric {col}'})
    invalid_mask |= bad_num

# Separate valid from invalid
cleaned_df = raw_df[~invalid_mask].copy()
error_df   = pd.DataFrame(validation_errors)

print(f'✓ Valid rows:    {len(cleaned_df)}')
print(f'✗ Invalid rows:  {len(raw_df[invalid_mask])} → written to error log')

if not error_df.empty:
    print('\nValidation errors:')
    display(error_df)

✓ Valid rows:    20
✗ Invalid rows:  0 → written to error log


---
## Stage 3: Data Transformation

Apply type casting, derive analytical fields, and flag anomalies. All transformations are deterministic and documented — this ensures reproducibility across pipeline runs.

In [5]:
# ── Type casting ──────────────────────────────────────────────
cleaned_df['service_date']  = pd.to_datetime(cleaned_df['service_date'])
cleaned_df['billed_amount'] = pd.to_numeric(cleaned_df['billed_amount'])
cleaned_df['paid_amount']   = pd.to_numeric(cleaned_df['paid_amount'])

# ── Date part extraction ──────────────────────────────────────
# Useful for partitioning and time-series aggregation downstream
cleaned_df['service_year']    = cleaned_df['service_date'].dt.year
cleaned_df['service_month']   = cleaned_df['service_date'].dt.month
cleaned_df['service_quarter'] = cleaned_df['service_date'].dt.quarter

# ── Derived metrics ───────────────────────────────────────────
# Payment rate = paid / billed — key managed care performance metric
cleaned_df['payment_rate'] = np.where(
    cleaned_df['billed_amount'] > 0,
    (cleaned_df['paid_amount'] / cleaned_df['billed_amount']).round(4),
    0.0
)

# ── Boolean flags for analytics layer ────────────────────────
cleaned_df['is_inpatient']  = cleaned_df['claim_type']    == 'IP'
cleaned_df['is_denied']     = cleaned_df['claim_status']  == 'DENIED'
cleaned_df['is_behavioral'] = cleaned_df['provider_type'] == 'BEHAVIORAL'

# ── Anomaly detection ─────────────────────────────────────────
# Denied claims with paid_amount > 0 — indicates a post-payment reversal
# that was not reconciled. Flagged for audit, not silently corrected.
cleaned_df['anomaly_paid_denied'] = cleaned_df['is_denied'] & (cleaned_df['paid_amount'] > 0)

n_anomalies = cleaned_df['anomaly_paid_denied'].sum()
if n_anomalies:
    print(f'⚠  Anomaly detected: {n_anomalies} denied claim(s) with paid_amount > 0 — flagged for review')
else:
    print('✓ No payment anomalies detected')

print(f'\nTransformed shape: {cleaned_df.shape}')
cleaned_df.head()

✓ No payment anomalies detected

Transformed shape: (20, 22)


,claim_id,member_id,state_code,plan_id,service_date,claim_type,diagnosis_code,procedure_code,billed_amount,paid_amount,claim_status,provider_id,provider_type,managed_care_flag,service_year,service_month,service_quarter,payment_rate,is_inpatient,is_denied,is_behavioral,anomaly_paid_denied
0,CLM0001,MBR1001,TX,MCO_TX_01,2024-01-05,IP,J18.9,99223,"12,500.00","9,800.00",PAID,PRV5001,HOSPITAL,Y,2024,1,1,0.78,True,False,False,False
1,CLM0002,MBR1002,TX,MCO_TX_01,2024-01-07,OP,Z00.00,99213,250.00,180.00,PAID,PRV5002,PHYSICIAN,Y,2024,1,1,0.72,False,False,False,False
2,CLM0003,MBR1003,FL,MCO_FL_02,2024-01-08,OP,E11.9,99214,350.00,270.00,PAID,PRV5003,PHYSICIAN,Y,2024,1,1,0.77,False,False,False,False
3,CLM0004,MBR1004,FL,MCO_FL_02,2024-01-10,IP,I21.9,99223,"18,000.00","14,200.00",PAID,PRV5004,HOSPITAL,Y,2024,1,1,0.79,True,False,False,False
4,CLM0005,MBR1005,CA,MCO_CA_03,2024-01-12,OP,F32.1,90837,200.00,150.00,PAID,PRV5005,BEHAVIORAL,Y,2024,1,1,0.75,False,False,True,False


---
## Stage 4: Curated Output

Produce the final curated dataset. In production this would write to a Delta table (Databricks) or partitioned Parquet in S3/GCS, partitioned by `state_code` and `service_year` for efficient downstream querying.

In [6]:
# Curated layer — clean, typed, enriched, ready for analytics
curated_df = cleaned_df[[
    'claim_id', 'member_id', 'state_code', 'plan_id',
    'service_date', 'service_year', 'service_month', 'service_quarter',
    'claim_type', 'diagnosis_code', 'procedure_code',
    'provider_id', 'provider_type', 'managed_care_flag',
    'billed_amount', 'paid_amount', 'payment_rate', 'claim_status',
    'is_inpatient', 'is_denied', 'is_behavioral', 'anomaly_paid_denied'
]].copy()

print(f'Curated rows:   {len(curated_df)}')
print(f'Curated cols:   {len(curated_df.columns)}')
curated_df.dtypes

Curated rows:   20
Curated cols:   22


claim_id                       object
member_id                      object
state_code                     object
plan_id                        object
service_date           datetime64[ns]
service_year                    int32
service_month                   int32
service_quarter                 int32
claim_type                     object
diagnosis_code                 object
procedure_code                 object
provider_id                    object
provider_type                  object
managed_care_flag              object
billed_amount                 float64
paid_amount                   float64
payment_rate                  float64
claim_status                   object
is_inpatient                     bool
is_denied                        bool
is_behavioral                    bool
anomaly_paid_denied              bool
dtype: object

In [7]:
# Pipeline summary — written alongside output for auditability
pipeline_summary = {
    'run_timestamp':   RUN_TIMESTAMP,
    'rows_raw':        row_count_raw,
    'rows_valid':      len(curated_df),
    'rows_invalid':    len(error_df),
    'anomalies_found': int(n_anomalies),
    'total_billed':    round(curated_df['billed_amount'].sum(), 2),
    'total_paid':      round(curated_df['paid_amount'].sum(), 2),
    'denial_rate_pct': round(curated_df['is_denied'].mean() * 100, 2),
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
curated_df.to_csv(f'{OUTPUT_DIR}/curated_claims.csv', index=False)
with open(f'{OUTPUT_DIR}/pipeline_run_summary.json', 'w') as f:
    json.dump(pipeline_summary, f, indent=2)
if not error_df.empty:
    error_df.to_csv(f'{OUTPUT_DIR}/validation_errors.csv', index=False)

print('Output written:')
print(f'  ✓ {OUTPUT_DIR}/curated_claims.csv')
print(f'  ✓ {OUTPUT_DIR}/pipeline_run_summary.json')
if not error_df.empty:
    print(f'  ✓ {OUTPUT_DIR}/validation_errors.csv')

print('\nPipeline summary:')
print(json.dumps(pipeline_summary, indent=2))

Output written:
  ✓ ../data/output/curated_claims.csv
  ✓ ../data/output/pipeline_run_summary.json

Pipeline summary:
{
  "run_timestamp": "2026-03-19T13:21:15.253491",
  "rows_raw": 20,
  "rows_valid": 20,
  "rows_invalid": 0,
  "anomalies_found": 0,
  "total_billed": 78150.0,
  "total_paid": 57720.0,
  "denial_rate_pct": 10.0
}


---
## Scalability: Python → PySpark

This notebook processes data using **Python/pandas**, which is ideal for:
- Local development and prototyping
- Datasets that fit in memory
- Fast iteration during development

For **production scale** (millions of claims, multi-state data), the same logic runs in PySpark (`pyspark/claims_transformation.py`):

| Aspect | Python (this notebook) | PySpark (production) |
|--------|----------------------|---------------------|
| Data size | Up to ~1GB | Terabytes |
| Processing | Single machine | Distributed cluster |
| Output | CSV | Partitioned Parquet / Delta |
| Platform | Local / any | Databricks / EMR |

The transformation logic is identical — only the execution engine changes.